In [1]:
##### Converts raw raster on irrigation land into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of cropland equiped for irrigation 

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [7]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# irrigation raster
irrigation = f"{cd}/Data/Raw/Predictors/Irrigated_area_Mehta/G_AEI_2015.ASC"

# cropland raster
cropland = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# save paths
pixels_pct_irr = f"{cd}/Data/Clean/Predictors/Rasters/pct_cropland_irrigated.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/percent_cropland_irrigated.csv"

In [6]:
#### Run resampling for pixel scale 

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

irr  = resample_to_ref(irrigation)
crop = resample_to_ref(cropland)


### CALCULATE 
# compute % irrigated
with np.errstate(invalid="ignore", divide="ignore"):
    pct_irrigated = np.where(
        (crop > 0) & ~np.isnan(crop) & ~np.isnan(irr),
        irr / crop,
        np.nan
    )

pct_irrigated = np.clip(pct_irrigated, 0, 1)

# save 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = pct_irrigated.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_pct_irr, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [10]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_irr  = rasterstats.zonal_stats(gdf_proj, irrigation, stats="sum", nodata=-9999)
zonal_crop = rasterstats.zonal_stats(gdf_proj, cropland,   stats="sum", nodata=-9999)

### compute share at vector level
irr_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_irr])
crop_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_crop])

with np.errstate(invalid="ignore", divide="ignore"):
    pct = np.where(
        (crop_sums > 0) & ~np.isnan(crop_sums) & ~np.isnan(irr_sums),
        irr_sums / crop_sums,
        np.nan
    )

result["pct_cropland_irrigated"] = np.clip(pct, 0, 1)

### save
result.to_csv(vectors, index=False)